In [ ]:
#| hide
import os
os.chdir('..')

# End to end walkthrough

> Detailed description of all the functionalities that MLForecast provides.

## Data setup

For this example we'll use a subset of the M4 hourly dataset. You can find the a notebook with the full dataset [here](https://www.kaggle.com/code/lemuz90/m4-competition).

In [ ]:
import random

from datasetsforecast.m4 import M4

In [ ]:
await M4.async_download('data', group='Hourly')
df, *_ = M4.load('data', 'Hourly')
uids = df['unique_id'].unique()
random.seed(0)
sample_uids = random.choices(uids, k=4)
df = df[df['unique_id'].isin(sample_uids)].reset_index(drop=True)
df['ds'] = df['ds'].astype('int')
df

## EDA

We'll take a look at our series to get ideas for transformations and features.

In [ ]:
import matplotlib.pyplot as plt

def plot(df, last_n=24 * 14):
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 6), gridspec_kw=dict(hspace=0.5))
    uids = df['unique_id'].unique()
    for i, (uid, axi) in enumerate(zip(uids, ax.flat)):
        legend = i % 2 == 0
        df[df['unique_id'].eq(uid)].tail(last_n).set_index('ds').plot(ax=axi, title=uid, legend=legend)

In [ ]:
plot(df)

We can use the `MLForecast.preprocess` method to explore different transformations. It looks like these series have a strong seasonality on the hour of the day, so we can subtract the value from the same hour in the previous day to remove it. This can be done with the `differences`.

In [ ]:
from mlforecast import MLForecast

In [ ]:
fcst = MLForecast(
    models=[],  # we're not interested in modeling yet
    freq=1,  # our series have integer timestamps, so we'll just add 1 in every timestep
    differences=[24],
)
prep = fcst.preprocess(df, id_col='unique_id', time_col='ds', target_col='y').reset_index()
prep

This has subtacted the lag 24 from each value, we can see what our series look like now.

In [ ]:
plot(prep)

## Adding features

### Lags

Looks like the seasonality is gone, we can now try adding some lag features.

In [ ]:
fcst = MLForecast(
    models=[],
    freq=1,
    differences=[24],
    lags=[1, 24],
)
prep = fcst.preprocess(df, id_col='unique_id', time_col='ds', target_col='y')
prep

In [ ]:
prep.drop(columns=['ds']).corr()['y']

### Lag transforms

Lag transforms are defined as a dictionary where the keys are the lags and the values are lists of functions that transform an array. These must be [numba](http://numba.pydata.org/) jitted functions (so that computing the features doesn't become a bottleneck). There are some implemented in the [window-ops package](https://github.com/jmoralez/window_ops) but you can also implement your own.

If the function takes two or more arguments you can either:

* supply a tuple (tfm_func, arg1, arg2, ...)
* define a new function fixing the arguments

In [ ]:
from numba import njit
from window_ops.expanding import expanding_mean
from window_ops.rolling import rolling_mean

In [ ]:
@njit
def rolling_mean_48(x):
    return rolling_mean(x, window_size=48)


fcst = MLForecast(
    models=[],
    freq=1,
    differences=[24],
    lag_transforms={
        1: [expanding_mean],
        24: [(rolling_mean, 48), rolling_mean_48],
    },
)
prep = fcst.preprocess(df, id_col='unique_id', time_col='ds', target_col='y')
prep

You can see that both approaches get to the same result, you can use whichever one you feel most comfortable with.

### Date features

If your time column is made of timestamps then it might make sense to extract features like week, dayofweek, quarter, etc. You can do that by passing a list of strings with [pandas time/date components](https://pandas.pydata.org/docs/user_guide/timeseries.html#time-date-components). You can also pass functions that will take the time column as input, as we'll show here.

In [ ]:
def hour_index(times):
    return times % 24

fcst = MLForecast(
    models=[],
    freq=1,
    differences=[24],
    date_features=[hour_index],
)
prep = fcst.preprocess(df, id_col='unique_id', time_col='ds', target_col='y')
prep

## Training

Once you've decided the features, transformations and models that you want to use you can use the `MLForecast.fit` method instead, which will do the preprocessing and then train the models.

In [ ]:
import lightgbm as lgb

In [ ]:
lgb_params = {
    'verbosity': -1,
    'num_leaves': 512,
}

fcst = MLForecast(
    models=[
        lgb.LGBMRegressor(**lgb_params),
        lgb.LGBMRegressor(**lgb_params, objective='quantile', alpha=0.75),
        lgb.LGBMRegressor(**lgb_params, objective='quantile', alpha=0.25),        
    ],
    freq=1,
    differences=[24],
    lags=[1, 24],
    lag_transforms={
        1: [expanding_mean],
        24: [(rolling_mean, 48)],
    },
    date_features=[hour_index],
)
fcst.fit(df, id_col='unique_id', time_col='ds', target_col='y')

This computed the features and trained three different models using them. We can now compute our forecasts.

## Forecasting

In [ ]:
preds = fcst.predict(48).reset_index()
preds.columns = ['unique_id', 'ds', 'avg', 'q75', 'q25']
preds

In [ ]:
import pandas as pd

In [ ]:
plot(pd.concat([df, preds]), last_n=24 * 7)

## Estimating model performance

### Cross validation

In order to get an estimate of how well our model will be when predicting future data we can perform cross validation, which consist on training a few models independently on different subsets of the data, using them to predict a validation set and measuring their performance.

Since our data depends on time, we make our splits by removing the last portions of the series and using them as validation sets. This process is implemented in `MLForecast.cross_validation`.

In [ ]:
fcst = MLForecast(
    models=lgb.LGBMRegressor(**lgb_params),
    freq=1,
    differences=[24],
    lags=[1, 24],
    lag_transforms={
        1: [expanding_mean],
        24: [(rolling_mean, 48)],
    },
    date_features=[hour_index],
)
cv_result = fcst.cross_validation(
    df,
    n_windows=4,  # number of models to train/splits to perform
    window_size=48,  # length of the validation set in each window
    id_col='unique_id',
    time_col='ds',
    target_col='y',
)
cv_result

In [ ]:
plot(cv_result.drop(columns='cutoff'))

We can compute the RMSE on each split.

In [ ]:
def evaluate_cv(df):
    return df['y'].sub(df['LGBMRegressor']).pow(2).groupby(df['cutoff']).mean().pow(0.5)

split_rmse = evaluate_cv(cv_result)
split_rmse

And the average RMSE across splits.

In [ ]:
split_rmse.mean()

You can quickly try different features and evaluate them this way. We can try removing the differencing and using an exponentially weighted average of the lag 1 instead of the expanding mean.

In [ ]:
from window_ops.ewm import ewm_mean

In [ ]:
fcst = MLForecast(
    models=lgb.LGBMRegressor(**lgb_params),
    freq=1,
    lags=[1, 24],
    lag_transforms={
        1: [(ewm_mean, 0.5)],
        24: [(rolling_mean, 48)],      
    },
    date_features=[hour_index],    
)
cv_result2 = fcst.cross_validation(
    df,
    n_windows=4,
    window_size=48,
    id_col='unique_id',
    time_col='ds',
    target_col='y',
)
evaluate_cv(cv_result2).mean()

### LightGBMCV

In the same spirit of estimating our model's performance, `LightGBMCV` allows us to train a few [LightGBM](https://github.com/microsoft/LightGBM) models on different partitions of the data. The main differences with `MLForecast.cross_validation` are:

* It can only train LightGBM models.
* It trains all models **simultaneously** and gives us per-iteration averages of the errors across the complete forecasting window, which allows us to find the best iteration.

In [ ]:
from mlforecast.lgb_cv import LightGBMCV

In [ ]:
cv = LightGBMCV(
    freq=1,
    differences=[24],
    lags=[1, 24],
    lag_transforms={
        1: [expanding_mean],
        24: [(rolling_mean, 48)],
    },
    date_features=[hour_index],
    num_threads=2,
)
cv_hist = cv.fit(
    df,
    n_windows=4,
    window_size=48,
    id_col='unique_id',
    time_col='ds',
    target_col='y',
    params=lgb_params,
    eval_every=5,
    early_stopping_evals=5,    
    compute_cv_preds=True,
)

As you can see this gives us the error by iteration (controlled by the `eval_every` argument) and performs early stopping (which can be configured with `early_stopping_evals` and `early_stopping_pct`). If you set `compute_cv_preds=True` the out-of-fold predictions are computed using the best iteration found and are saved in the `cv_preds_` attribute.

In [ ]:
cv.cv_preds_

In [ ]:
plot(cv.cv_preds_.reset_index().drop(columns='window'))

You can use this class to quickly try different configurations of features and hyperparameters.